# HGP: matrices, logical operators, and the distance argument

Paper references: `sec:HGP` and `eq:HGP`. All ranks are computed over GF(2).

Companion paper: Yuan Yao, Guo Zhang, Yingli Yang, Yuanye Zhu, Xiaosi Xu, Xiao Yuan, and Ying Li, **“An introduction to qLDPC codes from a structural perspective.”** Submitted.

In [1]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / "qldpc_generator").is_dir():
    root = root.parent
sys.path.insert(0, str(root))
import numpy as np
from qldpc_generator.algebra import x, y


The common interface returns check rows and qubit columns for both matrices. $$H_XH_Z^{\mathsf T}=0,\quad K=N-\mathrm{rank}_2H_X-\mathrm{rank}_2H_Z.\tag{1}$$

In [2]:
from qldpc_generator import hgp
from qldpc_generator.certificates import hgp_distance_witness
from qldpc_generator.representations import tanner_edges, tanner_matrix
H=np.array([[1,1,0],[0,1,1]],dtype=np.uint8)
code=hgp(H,H)
assert code.parameters()['k']==1
assert np.array_equal(tanner_matrix(code.hx.shape,tanner_edges(code.hx)),code.hx)
code.parameters()

{'n': 13,
 'k': 1,
 'rank_x': 6,
 'rank_z': 6,
 'checks_x': 6,
 'checks_z': 6,
 'max_row_weight': 4,
 'max_column_weight': 2}

Choose a minimum-weight classical codeword $c$ and a coordinate vector $e_j$ outside the row space, and construct $$z=(0,c\otimes e_j).\tag{2}$$ The quotient map kills the second block of every Z stabilizer but not this candidate. The function checks these conditions and returns an upper-bound certificate.

In [3]:
cert=hgp_distance_witness(H)
assert code.is_logical(cert['logical'],'Z')
print('coordinate:', cert['coordinate'], 'logical support:', np.flatnonzero(cert['logical']).tolist())
print('upper bound:', cert['upper_bound'])
result=code.distance(max_weight=3)
assert result['status']=='exact' and result['value']==3
result

coordinate: 0 logical support: [4, 7, 10]
upper bound: 3


{'status': 'exact',
 'value': 3,
 'sectors': {'X': {'status': 'exact',
   'value': 3,
   'lower_bound': 3,
   'witness': [4, 5, 6],
   'subsets': 57},
  'Z': {'status': 'exact',
   'value': 3,
   'lower_bound': 3,
   'witness': [4, 7, 10],
   'subsets': 59}}}

In [4]:
LX,LZ=code.logical_operators()
assert np.array_equal(LX @ LZ.T % 2,np.eye(code.k,dtype=int))
print('LX=',LX.tolist(), 'LZ=',LZ.tolist())

LX= [[0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0]] LZ= [[0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0]]


Exhaustive search certifies distance three for this finite example. General HGP asymptotic distance statements depend on the cited theorem and the properties of the input codes; this notebook does not prove them. See README for the legacy class-interface conventions.